# Phase 2: ML Attrition Model — Step 2.1: Feature Engineering

This notebook prepares the feature matrix for our attrition models. We will:
1. Check and handle missing values.
2. Engineer four business-relevant features: `income_per_year_at_company`, `years_since_last_promotion_gap`, `overall_satisfaction_composite`, and `experience_ratio`.
3. Drop constant columns, unique identifiers, and target/leakage variables.
4. Encode categorical variables using One-Hot Encoding.
5. Scale numeric features using `StandardScaler`.
6. Save the resulting feature matrix to `data/processed/feature_matrix.csv`.

In [1]:
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

proc_dir = os.path.join("data", "processed")
print(f"Processed directory: {os.path.abspath(proc_dir)}")

Processed directory: C:\Users\Harshit Mishra\OneDrive\Desktop\enterprise_hr_ai\data\processed


## 1. Load Processed Dataset

In [2]:
df_emp = pd.read_csv(os.path.join(proc_dir, "employees.csv"))
print(f"Dataset Shape: {df_emp.shape}")

Dataset Shape: (1470, 35)


## 2. Handle Missing Values
We inspect the dataset for any missing values.

In [3]:
missing_count = df_emp.isnull().sum().sum()
print(f"Total missing values: {missing_count}")
if missing_count > 0:
    # Fill numeric nulls with median, categorical with mode
    for col in df_emp.columns:
        if df_emp[col].isnull().sum() > 0:
            if df_emp[col].dtype in [np.float64, np.int64]:
                df_emp[col] = df_emp[col].fillna(df_emp[col].median())
            else:
                df_emp[col] = df_emp[col].fillna(df_emp[col].mode()[0])
    print("Handled missing values.")
else:
    print("No missing values found. Imputation skipped.")

Total missing values: 0
No missing values found. Imputation skipped.


## 3. Engineer Features

We engineer four key features:
1. **`income_per_year_at_company`**: MonthlyIncome * 12 / (YearsAtCompany + 1). Captures under-compensation relative to tenure.
2. **`years_since_last_promotion_gap`**: YearsAtCompany - YearsSinceLastPromotion. Captures how long the employee worked prior to promotion, indicating stagnation if they've been at the company long but were promoted early.
3. **`overall_satisfaction_composite`**: Average of EnvironmentSatisfaction, JobSatisfaction, RelationshipSatisfaction, and WorkLifeBalance. A unified indicator of overall sentiment.
4. **`experience_ratio`**: YearsAtCompany / (TotalWorkingYears + 1). Measures the fraction of the career spent at this company, reflecting loyalty and career stability.

In [4]:
df_emp["income_per_year_at_company"] = (df_emp["MonthlyIncome"] * 12) / (df_emp["YearsAtCompany"] + 1.0)
df_emp["years_since_last_promotion_gap"] = df_emp["YearsAtCompany"] - df_emp["YearsSinceLastPromotion"]
df_emp["overall_satisfaction_composite"] = (
    df_emp["EnvironmentSatisfaction"] 
    + df_emp["JobSatisfaction"] 
    + df_emp["RelationshipSatisfaction"] 
    + df_emp["WorkLifeBalance"]
) / 4.0
df_emp["experience_ratio"] = df_emp["YearsAtCompany"] / (df_emp["TotalWorkingYears"] + 1.0)

print("Engineered features successfully. Preview:")
print(df_emp[["income_per_year_at_company", "years_since_last_promotion_gap", "overall_satisfaction_composite", "experience_ratio"]].head())

Engineered features successfully. Preview:
   income_per_year_at_company  ...  experience_ratio
0                10273.714286  ...          0.666667
1                 5596.363636  ...          0.909091
2                25080.000000  ...          0.000000
3                 3878.666667  ...          0.888889
4                13872.000000  ...          0.285714

[5 rows x 4 columns]


## 4. Drop Constants, Unique Identifiers, and Target
We drop columns with zero variance (`EmployeeCount`, `Over18`, `StandardHours`), the employee unique identifier (`EmployeeNumber`), and we separate the target column (`Attrition`) to prevent target leakage.

In [5]:
constant_cols = ["EmployeeCount", "Over18", "StandardHours"]
id_cols = ["EmployeeNumber"]
target_col = "Attrition"

# Separate target
y = df_emp[target_col].map({"Yes": 1, "No": 0})

# Drop from features dataframe
X = df_emp.drop(columns=constant_cols + id_cols + [target_col])
print(f"Features Shape after dropping constants/leakage: {X.shape}")

Features Shape after dropping constants/leakage: (1470, 34)


## 5. Encode Categorical Variables
We use One-Hot Encoding to convert categorical features into numeric flags.

In [6]:
categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()
print(f"Categorical columns to encode: {categorical_cols}")

# One-Hot Encode
X_encoded = pd.get_dummies(X, columns=categorical_cols, drop_first=True)
# Convert bool fields to float/int
bool_cols = X_encoded.select_dtypes(include=['bool']).columns
X_encoded[bool_cols] = X_encoded[bool_cols].astype(int)
print(f"Features Shape after encoding: {X_encoded.shape}")
print(X_encoded.head())

Categorical columns to encode: ['BusinessTravel', 'Department', 'EducationField', 'Gender', 'JobRole', 'MaritalStatus', 'OverTime']
Features Shape after encoding: (1470, 48)
   Age  DailyRate  ...  MaritalStatus_Single  OverTime_Yes
0   41       1102  ...                     1             1
1   49        279  ...                     0             0
2   37       1373  ...                     1             1
3   33       1392  ...                     0             1
4   27        591  ...                     0             0

[5 rows x 48 columns]


## 6. Scale Numeric Features
We scale all features using a `StandardScaler` to ensure our models (like Logistic Regression) converge and perform correctly.

In [7]:
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X_encoded), columns=X_encoded.columns)

print("Scaled features preview:")
print(X_scaled.head())

Scaled features preview:
        Age  DailyRate  ...  MaritalStatus_Single  OverTime_Yes
0  0.446350   0.742527  ...              1.458650      1.591746
1  1.322365  -1.297775  ...             -0.685565     -0.628241
2  0.008343   1.414363  ...              1.458650      1.591746
3 -0.429664   1.461466  ...             -0.685565      1.591746
4 -1.086676  -0.524295  ...             -0.685565     -0.628241

[5 rows x 48 columns]


## 7. Save Feature Matrix and Target
We join the scaled features and the target back together and save them to `data/processed/feature_matrix.csv`.

In [8]:
# Add target back as the first column
feature_matrix = X_scaled.copy()
feature_matrix["Attrition"] = y

output_path = os.path.join(proc_dir, "feature_matrix.csv")
feature_matrix.to_csv(output_path, index=False)
print(f"Successfully saved feature matrix to {output_path} with shape {feature_matrix.shape}")

Successfully saved feature matrix to data\processed\feature_matrix.csv with shape (1470, 49)
